# final_v2_run — bài nộp đề thi: BM25 chọn đoạn + gộp 1800, M=20, K=20

**CHẠY HAI LẦN.** Mỗi lần ~5,0h, dư 2,4x so với trần 12h. GPU T4, Internet On,
**Save & Run All (Commit)**.

Cấu hình đã đo trên dev300 18/08: `max n=2` = **0.9183**, hơn mốc cũ 0.9083 **+1,00**,
**thắng ở cả bốn giá trị n** (+2,34 / +1,00 / +1,00 / +0,67).

### Vì sao K=20 chứ không hạ xuống 15

`ce_deep` là **max trên K đoạn** → bớt đoạn thì mọi văn bản chỉ có thể TỤT, không bao
giờ tăng. Hạ K là đòn **tiết kiệm**, không phải đòn **điểm**.

Đo trên oracle: **3/24 văn bản có đoạn thắng nằm ở hạng BM25 16-20**, cắt xuống 15 là
mất thật (ca nặng nhất mất 0,478 điểm CE). Recall@5 tình cờ không đổi vì cả ba ca đó
đằng nào cũng thua — **may, không phải cơ chế**. Và oracle chỉ có 24 văn bản đang HỤT
top-5; **275 gold đang THẮNG thì không có dữ liệu nào** cho biết chúng có phụ thuộc
đoạn hạng 16-20 hay không.

→ Quota 26h thì không có lý do đánh đổi điểm lấy giờ. Giữ K=20, tách hai lượt.

### Tách bằng `skip` — cơ chế đã chứng minh trên đề thi thật 17/08

| | `SKIP, M_DOC` | `SCORES_IN` | ra file |
|---|---|---|---|
| **lượt A** | `0, 10` | `scores_public.json` | `scores_public_bm25merge_M10_K20.json` |
| **lượt B** | `10, 20` | file của lượt A | `scores_public_bm25merge_M20_K20.json` |

Hợp lệ vì hai lượt dùng **chung** `MERGE_CHARS=1800` và **chung** `pick_chunks` —
cảnh báo trong docstring `deepen_one` là về việc TRỘN hai cấu hình khác nhau, không
phải tách một cấu hình. Thứ hạng sort theo `ce` (tầng 1, không bao giờ bị ghi đè) nên
không đổi giữa hai lượt. Lượt A ngày 17/08 giữ nguyên tuyệt đối `ce_deep` hạng 1-10
qua hai phiên GPU riêng biệt: **0 lệch**.

**Lượt A tự sinh bài nộp được** (M=10) — dùng làm bảo hiểm nếu lượt B trục trặc.

### Upload lên dataset `project-ir`

| trước lượt | file |
|---|---|
| **A** | `deep_chunk.py` (bản mới) · `Ketqua_E/scores_public.json` (6,8MB) |
| **B** | `scores_public_bm25merge_M10_K20.json` (output lượt A) |

**Thiếu file đầu vào là tầng 1 chạy lại, mất 3,4h vô ích.**
`submission.py`, `rerank.py`, `rerank_from_d.py`, `make_candidates_fallback.py`,
`public-official.json`, `dev_1000_locked.json`, `selected-contexts/` đã có sẵn.

### Ba thứ lượt này KHÔNG cần

1. **Không chạy tầng 1** — đọc file scores đã có.
2. **Không đọc `bm25_top100_public.json` (335MB)** — thứ tự BM25 suy từ khoá `bm25`
   trong chính file scores. Đã kiểm 17/08: dựng lại ra đúng 0.8883/0.9083 trên dev300.
   Tiết kiệm ~2GB RAM, đúng lúc `_cache` phình vì gộp đoạn.
3. **Không cần `metrics.py`** — đề thi `answer: null`, không đo được gì.

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
INPUT_DIR = "/kaggle/input/project-ir"     # đổi đúng slug — xem output bên dưới
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, time, zipfile
from pathlib import Path

from submission import build_submission, save_submission_zip, validate_submission_file
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
import deep_chunk as DC

# ============ ĐỔI ĐÚNG HAI DÒNG NÀY GIỮA HAI LƯỢT ============
SKIP, M_DOC = 0, 10                                  # lượt B:  10, 20
SCORES_IN = f"{INPUT_DIR}/scores_public.json"        # lượt B:  f"{INPUT_DIR}/scores_public_bm25merge_M10_K20.json"
# =============================================================

K_CHUNK, TOPK = 20, 5
CKPT_EVERY = 100                                     # lưu tạm mỗi N câu
TEST_Q   = f"{INPUT_DIR}/public-official.json"       # 1000 câu, answer=null
DEV_GOLD = f"{INPUT_DIR}/dev_1000_locked.json"       # CHỈ để guard chống nộp nhầm dev
CTX_DIR  = f"{INPUT_DIR}/selected-contexts"
OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DC.MERGE_CHARS = 1800                                # ĐẶT TRƯỚC mọi lời gọi — nó đổi cache
assert hasattr(DC, "MERGE_CHARS"), "deep_chunk.py trên dataset là BẢN CŨ — upload lại"
assert os.path.isdir(CTX_DIR),    f"KHÔNG thấy {CTX_DIR}"
assert os.path.isfile(SCORES_IN), f"CHƯA UPLOAD {SCORES_IN} — thiếu là mất 3,4h chạy lại tầng 1"
print(f"LƯỢT {'A' if SKIP == 0 else 'B'} | SKIP={SKIP} M_DOC={M_DOC} K={K_CHUNK} "
      f"MERGE_CHARS={DC.MERGE_CHARS}")
print(f"đầu vào: {SCORES_IN}")

## Bước 0 — DẤU VÂN TAY FILE CODE, in thẳng vào log

Kaggle nạp code từ dataset, mà dataset thì dễ quên upload đè. Cell này in **hash + kích
thước + giờ sửa** của mọi file `.py` lượt này phụ thuộc, rồi kiểm **hành vi** của
`pick_chunks`, rồi chạy luôn self-test có sẵn trong `deep_chunk.py`.

**Đối chiếu với bản đã kiểm trên máy 18/08:**

| file | bytes | sha256 (12 ký tự đầu) |
|---|---|---|
| **`deep_chunk.py`** | **14.621** | **`2a8c193a1e4a`** ← bản BM25 + MERGE_CHARS |
| `rerank_from_d.py` | 12.422 | `8688f4b7308d` |
| `submission.py` | 16.732 | `9734ddd4255d` |
| `make_candidates_fallback.py` | 14.010 | `30bd05a6eef8` |
| `scores_public.json` | 6.806.629 | `4a752fc89cb9` |
| `public-official.json` | 186.517 | `adaa250c5469` |

`deep_chunk.py` ra hash khác hoặc 8.659 bytes = **bản CŨ**, dừng lại upload đè.

Hash chỉ để đối chiếu. Thứ thật sự chặn lỗi là bốn `assert` hành vi bên dưới — hash lạ mà
logic đúng thì vẫn chạy được, còn assert kiểm đúng thứ cần. Self-test cuối cùng là chốt
mạnh nhất: nó chạy chính `__main__` của `deep_chunk.py`, trong đó có ca kiểm BM25 phải
chọn đoạn chứa term hiếm thay vì đoạn lặp nhiều term rác — ca mà bản `count` cũ trượt.


In [ ]:
# ---------- dấu vân tay: file nào, bản nào ----------
import hashlib, inspect, subprocess, datetime

for mod in (DC, sys.modules["rerank_from_d"], sys.modules["submission"],
            sys.modules["make_candidates_fallback"]):
    f = inspect.getsourcefile(mod)
    b = open(f, "rb").read()
    t = datetime.datetime.fromtimestamp(os.path.getmtime(f))
    print(f"  {os.path.basename(f):28s} {len(b):>7,}B  "
          f"sha256:{hashlib.sha256(b).hexdigest()[:12]}  sửa {t:%d/%m %H:%M}")

for f in (SCORES_IN, TEST_Q):
    b = open(f, "rb").read()
    print(f"  {os.path.basename(f):28s} {len(b):>7,}B  "
          f"sha256:{hashlib.sha256(b).hexdigest()[:12]}")

# ---------- kiểm HÀNH VI: quan trọng hơn hash ----------
src = inspect.getsource(DC.pick_chunks)
assert hasattr(DC, "MERGE_CHARS"), \
    "deep_chunk.py là BẢN CŨ (không có MERGE_CHARS) — upload đè lên dataset rồi chạy lại"
assert "idf.get" in src, \
    "pick_chunks chưa phải bản BM25 — upload đè deep_chunk.py"
assert "len(qs & p[1])" not in src, \
    "pick_chunks VẪN là bản đếm từ trùng cũ — upload đè deep_chunk.py"
assert (DC.K1, DC.B) == (1.5, 0.75), f"tham số BM25 bị vặn: K1={DC.K1} B={DC.B}"
print("\n  pick_chunks = BM25 mức đoạn  OK")
print(f"  MERGE_CHARS = {DC.MERGE_CHARS}  (0 = chưa đặt, PHẢI là 1800)")
assert DC.MERGE_CHARS == 1800, "quên đặt DC.MERGE_CHARS = 1800 ở cell config"

# ---------- self-test có sẵn trong deep_chunk.py ----------
r = subprocess.run([sys.executable, inspect.getsourcefile(DC)], capture_output=True, text=True)
print(f"  self-test: {r.stdout.strip() or r.stderr.strip()[-200:] or '(không in gì — bản cũ?)'}")
assert r.returncode == 0, "self-test của deep_chunk.py THẤT BẠI — dừng lại"

print(f"\n  LƯỢT {'A' if SKIP == 0 else 'B'} | SKIP={SKIP} M_DOC={M_DOC} "
      f"K_CHUNK={K_CHUNK} MERGE_CHARS={DC.MERGE_CHARS} | n_bm25 sinh cả 2 và 0")


## Bước 1 — Đọc đề thi + scores đầu vào, GUARD chống nộp nhầm dev

Bài nộp 09/08 bị BTC từ chối vì nộp dự đoán của 150 câu dev — định dạng đúng hoàn toàn,
chỉ sai bộ `question_id`. dev1000 bao trùm dev300 + dev150 nên chặn được hết.

Kèm chốt chặn quan trọng nhất của cơ chế `skip`: **lượt A phải nhận tầng 1 SẠCH,
lượt B phải nhận đúng 10 văn bản/câu đã có `ce_deep`.** Sai là dừng ngay, đừng đốt 5h.

In [ ]:
raw = json.load(open(TEST_Q, encoding="utf-8-sig"))
test_q = {str(k): (v["question"] if isinstance(v, dict) else v) for k, v in raw.items()}
scores = {str(k): v for k, v in json.load(open(SCORES_IN, encoding="utf-8")).items()}

dev_ids = {str(x) for x in json.load(open(DEV_GOLD, encoding="utf-8"))}
trung = set(test_q) & dev_ids
assert dev_ids, "dev_1000_locked.json RỖNG — guard vô nghĩa, dừng lại"
assert not trung, f"CHẶN CỨNG: {len(trung)} qid trùng dev — đang chạy nhầm bộ câu hỏi"
assert set(scores) == set(test_q), (
    f"scores không khớp đề thi (thiếu {len(set(test_q)-set(scores))}, "
    f"thừa {len(set(scores)-set(test_q))})")
assert all(scores[q] for q in test_q), "có câu điểm RỖNG"

nd = [sum(1 for v in scores[q].values() if "ce_deep" in v) for q in test_q]
if SKIP == 0:
    assert max(nd) == 0, (
        f"LƯỢT A phải nhận tầng 1 SẠCH, nhưng {sum(1 for x in nd if x)} câu đã có ce_deep. "
        f"Trỏ nhầm file? ce_deep của cách băm CŨ trộn vào là hỏng cả bài.")
else:
    assert min(nd) == max(nd) == SKIP, (
        f"LƯỢT B phải nhận output lượt A với ĐÚNG {SKIP} văn bản/câu có ce_deep, "
        f"đang thấy {min(nd)}-{max(nd)}. Trỏ nhầm file.")
    print(f"đầu vào lượt A hợp lệ: {SKIP}/{SKIP} văn bản/câu có ce_deep, cả {len(nd)} câu")

bm25 = {q: sorted(scores[q], key=lambda d: -scores[q][d]["bm25"]) for q in test_q}
print(f"{len(test_q)} câu đề thi | {len(scores[next(iter(test_q))])} văn bản/câu")
print(f"guard dev: {len(dev_ids):,} qid trong dev1000, giao với đề thi = {len(trung)} "
      f"(phải 0) | corpus {len(os.listdir(CTX_DIR)):,} file")

## Bước 2 — ĐẾM trước khi chấm (quy tắc 6)

Mong đợi **~178.000 đoạn** mỗi lượt. Ra ~355.000 là `SKIP` chưa ăn (đang chấm cả 20) —
**dừng ngay**, đó đúng lỗi đã suýt dính hôm 17/08. Băm 1000 câu mất vài phút, kiên nhẫn.

In [ ]:
t0 = time.time()
n2 = DC.count_deep_chunks(test_q, scores, CTX_DIR, M_DOC, K_CHUNK, skip=SKIP)
print(f"tầng 2 sẽ chấm {n2:,} đoạn ({n2/len(test_q):.0f}/câu) | băm+đếm {time.time()-t0:.0f}s")
print(f"ước {n2/10.0/3600:.1f}h ở nhịp 10,0 đoạn/s (đo được ở lượt dev 18/08)")
print(f"     {n2/5.0/3600:.1f}h nếu GPU chậm gấp đôi  <- vẫn phải < 12h")
assert 120_000 < n2 < 260_000, (
    f"{n2:,} lệch xa 178.000 — SKIP/M_DOC/MERGE_CHARS sai. "
    f"~355.000 nghĩa là SKIP chưa ăn.")
print(f"\n_cache đang giữ {len(DC._cache):,} văn bản")

In [ ]:
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda")

## Bước 3 — Tầng 2, LƯU TẠM mỗi 100 câu

Lượt 15/08 bị cắt ở 12h và mất trắng tầng 2. Checkpoint để nếu lại bị cắt thì phần đã
chấm vẫn dùng được — chạy lại notebook là nó tự đọc `ckpt` và làm nốt phần thiếu.

In [ ]:
CKPT = f"{OUTPUT_DIR}/ckpt_M{M_DOC}_K{K_CHUNK}.json"
deep = json.load(open(CKPT, encoding="utf-8")) if os.path.exists(CKPT) else {}
con_lai = [q for q in test_q if q not in deep]
print(f"đã có {len(deep)} câu | còn {len(con_lai)} câu")

t0 = time.time()
for i in range(0, len(con_lai), CKPT_EVERY):
    lo = con_lai[i:i + CKPT_EVERY]
    deep.update(DC.deepen_all({q: test_q[q] for q in lo}, scores, CTX_DIR,
                              score_fn, M_DOC, K_CHUNK, every=50, skip=SKIP))
    json.dump(deep, open(CKPT, "w", encoding="utf-8"), ensure_ascii=False)
    el, xong = time.time() - t0, len(deep)
    moi = xong - (len(test_q) - len(con_lai))          # số câu CHẤM ĐƯỢC trong lượt này
    con = (el / moi) * (len(test_q) - xong) / 60 if moi else 0
    print(f"[ckpt] {xong}/{len(test_q)} câu | {el/60:.0f} phút | còn ~{con:.0f} phút", flush=True)

el = time.time() - t0
print(f"\ntầng 2 xong trong {el/3600:.2f}h | nhịp thật {n2/el:.1f} đoạn/s (dev300 đo được 10,0)")

p = f"{OUTPUT_DIR}/scores_public_bm25merge_M{M_DOC}_K{K_CHUNK}.json"
json.dump(deep, open(p, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p} — TẢI VỀ TRƯỚC TIÊN (quy tắc 2)")
if SKIP == 0:
    print("     ^ ĐÂY LÀ ĐẦU VÀO CỦA LƯỢT B. Upload lên dataset trước khi chạy lượt B.")

## Bước 4 — Kiểm chéo cơ chế `skip`

Đúng năm phép kiểm đã dùng hôm 17/08. Quan trọng nhất là phép số 2: **lượt B không được
phép làm xê dịch `ce_deep` hạng 1-10 của lượt A dù chỉ một số.**

In [ ]:
nd2 = [sum(1 for v in deep[q].values() if "ce_deep" in v) for q in test_q]
print(f"1. văn bản có ce_deep mỗi câu: {min(nd2)}-{max(nd2)} (phải đúng {M_DOC})"
      + ("  OK" if min(nd2) == max(nd2) == M_DOC else "  <-- SAI"))
assert min(nd2) == max(nd2) == M_DOC

lech_ce = sum(1 for q in test_q for d in deep[q] if deep[q][d]["ce"] != scores[q][d]["ce"])
print(f"2. `ce` tầng 1 bị ghi đè: {lech_ce} cặp (phải 0)" + ("  OK" if not lech_ce else "  <-- SAI"))
assert not lech_ce

if SKIP:
    lech_deep = sum(1 for q in test_q for d in scores[q]
                    if "ce_deep" in scores[q][d] and deep[q][d].get("ce_deep") != scores[q][d]["ce_deep"])
    print(f"3. `ce_deep` hạng 1-{SKIP} của lượt A bị xê dịch: {lech_deep} (phải 0)"
          + ("  OK" if not lech_deep else "  <-- SAI, skip hỏng"))
    assert not lech_deep
else:
    print("3. (bỏ qua — lượt A không có ce_deep cũ để so)")

print(f"4. {len(deep)} câu, khớp khít bộ câu hỏi: {set(deep) == set(test_q)}")
assert set(deep) == set(test_q)

## Bước 5 — Sinh HAI bài nộp: n=2 (chốt) và n=0 (đo cơ chế)

0 giờ GPU thêm. `n=2` là bài chốt, đã xác nhận bằng public LB (0.8573 vs 0.8350/0.8532/0.8424).

`n=0` là **phép đo cơ chế**: dev300 mới cho `max n=0` = 0.9217 > `n=2` = 0.9183, tức đỉnh
vừa dời. Chênh chỉ 1 câu nên dev300 không kết luận được, nhưng cơ chế hợp lý —
`blend_bm25_first` ép 2 suất vì reranker hay vứt nhầm thứ BM25 chọn đúng; reranker khoẻ
lên thì giá trị cứu của blend giảm, cái giá 2 suất thì không đổi. **Nộp n=2 TRƯỚC.**

In [ ]:
expected = set(test_q)
fallback = {q: list(bm25[q]) for q in test_q}          # cả 100 id, để bù cho đủ 5
zips = {}

for n in (2, 0):
    pred = {q: blend_bm25_first(DC.rank_by(deep[q], "max"), bm25[q], k=TOPK, n_bm25=n)
            for q in test_q}
    d = f"{OUTPUT_DIR}/n{n}"
    Path(d).mkdir(exist_ok=True)
    sub = build_submission(pred, ranked_fallback_dict=fallback, k=TOPK, expected_qids=expected)
    z = save_submission_zip(sub, out_dir=d)
    errs = validate_submission_file(os.path.join(d, "submission.json"),
                                    expected_qids=expected, k=TOPK)
    moi = f"{OUTPUT_DIR}/submission_v2_M{M_DOC}_n{n}.zip"
    os.replace(z, moi)
    zips[n] = moi
    print(f"n={n}: {'LỖI -> ' + '; '.join(errs) if errs else 'OK'}  {os.path.basename(moi)}")
    assert not errs, f"n={n} có lỗi, KHÔNG ĐƯỢC NỘP"

p2 = {q: blend_bm25_first(DC.rank_by(deep[q], "max"), bm25[q], TOPK, 2) for q in test_q}
p0 = {q: blend_bm25_first(DC.rank_by(deep[q], "max"), bm25[q], TOPK, 0) for q in test_q}
print(f"\nn=2 vs n=0 lệch {sum(1 for q in test_q if set(p2[q]) != set(p0[q]))}/1000 câu")

## Bước 6 — Soi lại zip TỪ ĐĨA, không tin biến trong bộ nhớ

In [ ]:
for n, z in zips.items():
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        assert names == ["submission.json"], f"n={n}: zip phải chứa DUY NHẤT submission.json, có {names}"
        raw_b = zf.read("submission.json")
    assert raw_b[:3] != b"\xef\xbb\xbf", f"n={n}: có BOM — BTC yêu cầu UTF-8 không BOM"
    data = json.loads(raw_b.decode("utf-8"))
    assert set(data) == expected, f"n={n}: bộ question_id không khớp đề thi"
    for q, v in data.items():
        a = v["answer"]
        assert isinstance(a, list) and 1 <= len(a) <= TOPK, f"{q}: có {len(a)} id"
        assert len(a) == len(set(a)), f"{q}: id trùng"
        assert all(isinstance(x, str) for x in a), f"{q}: id không phải string"
    du5 = sum(1 for v in data.values() if len(v["answer"]) == TOPK)
    print(f"n={n}: {len(data)} câu, {du5} câu đủ 5 id — SẠCH")

print("\n" + "=" * 70)
if SKIP == 0:
    print("LƯỢT A XONG. Việc phải làm, theo thứ tự:")
    print("  1. Tải outputs/ về máy")
    print(f"  2. Upload scores_public_bm25merge_M{M_DOC}_K{K_CHUNK}.json lên dataset project-ir")
    print("  3. Sửa cell config: SKIP, M_DOC = 10, 20  và  SCORES_IN trỏ file vừa upload")
    print("  4. Chạy lượt B")
    print(f"  (submission_v2_M{M_DOC}_n2.zip nộp được ngay — bảo hiểm nếu lượt B trục trặc)")
else:
    print("XONG CẢ HAI LƯỢT. NỘP submission_v2_M20_n2.zip TRƯỚC.")
    print("Mốc phải vượt: 0.8840 (bài B M=20 cấu hình cũ).")
    print("dev300 nói cấu hình này +1,00 -> ước public ~0.894.")
    print("Dưới 0.884 là TỤT: quay về bài cũ, đừng cố cứu.")
    print("Rồi mới nộp submission_v2_M20_n0.zip — CHỈ để đo blend còn đáng không.")
print("=" * 70)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        print(f"  {f}  {os.path.getsize(fp):,} bytes")
print("\nTẢI TOÀN BỘ outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN")